# Screen sample repetitions

Finds samples sequenced in more than one run in a Mad4HaTter allele table and reports their depth in each run. It can also write an allele table in which those samples are merged, cut to their best run, or renamed. This is the notebook version of `screen_sample_repetitions.py`, and both give the same output files.

**How to use:** edit the **Settings** cell, then run all cells (**Run All**). Relative paths are read from the folder of this notebook.

## Prerequisites

- Python 3.9 or newer
- pandas (`pip install pandas` or `conda install pandas`)
- Jupyter: JupyterLab, Jupyter Notebook or VS Code

## Settings

**Allele table (`ALLELE_FILE`):** a `.txt`, `.tsv` or `.csv` file, optionally `.gz`. Column names are matched ignoring case.

- **Required columns:** the sample ID (`SampleID`, `sample_id`, `sample_name` or `sample`), `locus` and `reads`.
- **`Run`:** tells which run each row comes from. Without it, each sequencing ID (for example `N2076817-1_S3_L001`) counts as its own run.
- **Optional columns:** `asv`, `NIDA`, `norm.reads.locus` and `n.alleles`.

**Manifests (`MANIFEST_FILES`):** one or more pipeline manifests, each with a `SampleID` or `sample_name` column and a `SampleType` column.

- **What gets screened:** only IDs with `SampleType = sample`. Controls and IDs not in a manifest are left unchanged.
- **Matching:** IDs are standardised first, so `N2076817-1_S3_L001`, `N2076817_1_S12`, `N2076817-1` and `2076817.1` are the same sample.

**Options:** set at most one of `MERGE`, `BEST_COVERAGE` and `KEEP_DUPS` to `True`.

- **Best run:** the run with the most loci above `MIN_DEPTH` reads. Ties go to the run with more reads.
- **`KEEP_DUPS`:** the letter goes after `_S*_L001` in the sample ID and at the end of `NIDA`, for example `N1910666-3_S59_L001_a` and `1910666.3_a`.
- **`MERGE`:** needs an `asv` column when the allele table is written, because allele names are numbered separately in each run.

In [ ]:
# ---- Input files -----------------------------------------------------------
ALLELE_FILE = "allele_table.txt"     # allele table (.txt, .tsv or .csv, optionally .gz)
MANIFEST_FILES = ["manifest.csv"]    # one or more, e.g. ["manifest_run1.csv", "manifest_run2.csv"]
ALLELES_ID_COL = None                # sample ID column, only if not SampleID, sample_id, sample_name or sample

# ---- Output files ----------------------------------------------------------
OUT_FILE = "screening.tsv"           # screening table (.csv gives commas, anything else tabs)
OUT_ALLELES_FILE = None              # allele table with the option below applied, e.g. "allele_table_best_run.txt"; None = not written

# ---- Samples in more than one run: set at most one to True ------------------
MERGE = False           # combine each sample's runs into one, adding up the reads
BEST_COVERAGE = False   # keep only the best run of each sample
KEEP_DUPS = False       # keep every run and add _a, _b, ... to the IDs (a = best run)

# ---- Screening -------------------------------------------------------------
MIN_DEPTH = 100         # reads a locus must exceed to count in Depth_Read_Loci
SHOW_ALL = False        # True: also list samples in only one run in the screening table

## Functions

These are the same functions as in `screen_sample_repetitions.py`.

In [ ]:
import gzip
import os
import re

import pandas as pd

# Column names are matched ignoring case (SampleID = sampleID, Reads = reads)
ID_NAMES = ["sampleid", "sample_id", "sample_name", "sample"]
TYPE_NAMES = ["sampletype", "sample_type"]


def shortlist(values, n=20):
    values = list(values)
    return ", ".join(map(str, values[:n])) + (" ..." if len(values) > n else "")


def find_col(df, wanted, names, label):
    if wanted:
        if wanted not in df.columns:
            raise ValueError(f"Column '{wanted}' not found for {label}. Columns: {list(df.columns)}")
        return wanted
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n in lower:
            return lower[n]
    return None


def normalise_id(raw):
    s = str(raw).strip()
    s = re.sub(r"__.*$", "", s)           # __MULE_NextSEQ03_300725
    s = re.sub(r"_S\d+(_L\d+)?$", "", s)  # _S12_L001
    m = re.match(r"^N?(\d+)(?:[-_.](\d+))?$", s)
    if m:
        base, dec = m.groups()
        return base if dec in (None, "0") else f"{base}.{dec}"
    return s.replace("-", "_")


def letter(rank):
    """0 -> a, 1 -> b, ..., 25 -> z, 26 -> aa."""
    s, n = "", rank + 1
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(97 + r) + s
    return s


def add_letter(raw, tag):
    """Put tag right after _S12 / _S3_L001 if the ID has it, otherwise at the end:
    N2076817_1_S64__HFS23_NextSEQ01_271025 -> N2076817_1_S64_a__HFS23_NextSEQ01_271025."""
    m = re.search(r"_S\d+(?:_L\d+)?(?=__|$)", raw)
    return raw[:m.end()] + tag + raw[m.end():] if m else raw + tag


def number_text(v):
    return str(int(v)) if float(v).is_integer() else str(float(v))


def separator(path):
    return "," if path.lower().removesuffix(".gz").endswith(".csv") else "\t"


def read_table(path):
    if path.lower().endswith((".xlsx", ".xls")):
        raise ValueError(f"{path}: save it as .csv or tab-separated .txt first.")
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="latin-1") as fh:
        first = fh.readline()
    sep = "\t" if "\t" in first else ","
    return pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False, encoding="latin-1")


def read_manifests(paths):
    """Standardised ID and SampleType (lower case) of every row of the manifests."""
    parts = []
    for path in paths:
        df = read_table(path)
        sid, typ = find_col(df, None, ID_NAMES, ""), find_col(df, None, TYPE_NAMES, "")
        if not (sid and typ):
            raise ValueError(f"{path}: the manifest needs a SampleID (or sample_name) column and a SampleType column. "
                     f"Columns: {list(df.columns)}")
        part = pd.DataFrame({"raw": df[sid].str.strip(), "SampleType": df[typ].str.strip().str.lower()})
        parts.append(part[part.raw != ""])
    man = pd.concat(parts).drop_duplicates()
    man["SampleID"] = man.raw.map(normalise_id)
    return man


def summarise(per_locus, keys, min_depth):
    """Reads, loci and deep loci for each group of per-locus read totals."""
    s = per_locus.groupby(keys).agg(
        Total_Reads=("reads", "sum"),
        Total_Loci=("reads", lambda v: int((v > 0).sum())),
        Depth_Read_Loci=("reads", lambda v: int((v > min_depth).sum()))).reset_index()
    s["Avg_Reads_per_Locus"] = (s.Total_Reads / s.Total_Loci.where(s.Total_Loci > 0)).round(2)
    return s


def resolve_alleles(al, c, links, repeated, ranks, mode):
    """Allele table with each sample in several runs merged, cut to its best run, or relabelled.

    c       column names in al: id, locus, reads, asv, run, nida, allele, norm, n_alleles
    links   row (index in al), SampleID and Run of every allele row of a screened sample
    repeated  rows left out when adding up reads (same sequence listed twice)
    ranks   SampleID, Run and Rank (0 = best run) of the samples in several runs
    """
    dup = links.merge(ranks, on=["SampleID", "Run"])
    if mode == "best_coverage":
        return al.drop(index=dup.row[dup.Rank > 0])

    if mode == "keep_dups":
        out = al.copy()
        tag = pd.Series(("_" + dup.Rank.map(letter)).to_numpy(), index=dup.row.to_numpy())
        for col in dict.fromkeys(x for x in (c["id"], c["nida"]) if x):
            vals = out.loc[tag.index, col]
            out.loc[tag.index, col] = [v if v == "" else add_letter(v, t) for v, t in zip(vals, tag)]
        return out

    # merge: one row per sample, locus and sequence; the other columns come
    # from the best run that has that sequence
    locus, reads = c["locus"], c["reads"]
    keys = ["_sid", locus, c["asv"]]
    use = dup[~dup.row.isin(repeated)]
    part = al.loc[use.row].assign(_sid=use.SampleID.to_numpy(), _rank=use.Rank.to_numpy())
    part["_row"] = part.index
    part["_reads"] = pd.to_numeric(part[reads], errors="coerce").fillna(0)
    part = part.sort_values(["_sid", "_rank", "_row"])
    part["_total"] = part.groupby(keys)._reads.transform("sum")
    # Keep each merged sample together, at the place of its first row
    part["_p"] = part.groupby("_sid")._row.transform("min")
    part["_l"] = part.groupby(["_sid", locus])._row.transform("min")
    m = part.drop_duplicates(keys).copy()

    m[reads] = m._total.map(number_text)
    at_locus = m.groupby(["_sid", locus])._total.transform("sum")
    if c["norm"]:
        m[c["norm"]] = (m._total / at_locus.where(at_locus > 0)).map(lambda v: "" if pd.isna(v) else str(v))
    if c["n_alleles"]:
        m[c["n_alleles"]] = m.groupby(["_sid", locus])._total.transform("size").astype(str)
    m[c["id"]] = m._sid.map(part[part._rank == 0].groupby("_sid")[c["id"]].first())
    if c["run"]:
        m[c["run"]] = m._sid.map(ranks.sort_values("Rank").groupby("SampleID").Run.agg(";".join))
    if c["allele"]:
        clash = m.duplicated(["_sid", locus, c["allele"]], keep=False)
        if clash.any():
            print(f"Note: {m[clash].groupby(['_sid', locus, c['allele']]).ngroups} allele names are used for two "
                  "different sequences in the same merged sample, because each run numbers its alleles "
                  "separately. Match alleles by asv, not by allele name.")

    rest = al.drop(index=dup.row)
    rest = rest.assign(_p=rest.index, _l=rest.index, _row=rest.index)
    out = pd.concat([rest, m]).sort_values(["_p", "_l", "_row"], kind="stable")
    return out[list(al.columns)]

## 1. Check the settings

In [ ]:
if isinstance(MANIFEST_FILES, str):
    MANIFEST_FILES = [MANIFEST_FILES]
modes = [name for name, on in [("merge", MERGE), ("best_coverage", BEST_COVERAGE), ("keep_dups", KEEP_DUPS)] if on]
if len(modes) > 1:
    raise ValueError("Set only one of MERGE, BEST_COVERAGE and KEEP_DUPS to True.")
mode = modes[0] if modes else None
if OUT_ALLELES_FILE and not mode:
    raise ValueError("OUT_ALLELES_FILE needs MERGE, BEST_COVERAGE or KEEP_DUPS set to True.")
inputs = {os.path.abspath(x) for x in [ALLELE_FILE, *MANIFEST_FILES]}
outputs = [os.path.abspath(x) for x in (OUT_FILE, OUT_ALLELES_FILE) if x]
if inputs & set(outputs) or len(set(outputs)) < len(outputs):
    raise ValueError("OUT_FILE and OUT_ALLELES_FILE must be different from each other and from the input files.")

## 2. Read the allele table

In [ ]:
al = read_table(ALLELE_FILE)
sid = find_col(al, ALLELES_ID_COL, ID_NAMES, "the allele table IDs")
locus = find_col(al, None, ["locus"], "")
reads = find_col(al, None, ["reads"], "")
run = find_col(al, None, ["run"], "")
asv = find_col(al, None, ["asv"], "")
if not (sid and locus and reads):
    raise ValueError(f"The allele table needs sample ID, locus and reads columns. Columns: {list(al.columns)}")
if OUT_ALLELES_FILE and mode == "merge" and not asv:
    raise ValueError("MERGE with OUT_ALLELES_FILE needs an asv column in the allele table to match alleles "
                     "across runs (allele names are numbered separately in each run).")

alleles = pd.DataFrame({"row": al.index, "raw": al[sid].str.strip(), "locus": al[locus],
                        "reads": pd.to_numeric(al[reads], errors="coerce").fillna(0)})
# Without a Run column, each sequencing ID (e.g. N2076817-1_S3_L001) counts as its own run
alleles["Run"] = al[run].str.strip() if run else alleles.raw
if asv:
    alleles["asv"] = al[asv]
alleles["SampleID"] = alleles.raw.map(normalise_id)
ids = set(alleles.SampleID)

## 3. Choose the samples from the manifests

Only IDs with `SampleType = sample` are screened.

In [ ]:
man = read_manifests(MANIFEST_FILES)
samples = set(man.SampleID[man.SampleType == "sample"])
mixed = sorted(samples & set(man.SampleID[man.SampleType != "sample"]))
not_in_manifest = sorted(ids - set(man.SampleID))
data = alleles[alleles.SampleID.isin(samples)]

links = data[["row", "SampleID", "Run"]]
repeated_rows = set()
if asv:
    # The same sequence listed twice under two allele names would be counted twice
    repeated = data.duplicated(["raw", "Run", "locus", "asv"])
    if repeated.any():
        print(f"Note: ignored {repeated.sum()} repeated sequence rows (same sample, run, locus and asv).")
    repeated_rows = set(data.row[repeated])
    data = data[~repeated]

## 4. Depth of each sample in each run

In [ ]:
per_locus = data.groupby(["SampleID", "Run", "locus"], as_index=False).reads.sum()
stats = summarise(per_locus, ["SampleID", "Run"], MIN_DEPTH)
raw_ids = data.groupby(["SampleID", "Run"]).raw.agg(lambda v: ";".join(sorted(set(v)))).rename("Raw_SampleID")
stats = stats.merge(raw_ids.reset_index(), on=["SampleID", "Run"])
stats["N_Runs"] = stats.groupby("SampleID").Run.transform("nunique")
# Best run first: most Depth_Read_Loci, then most Total_Reads
stats = stats.sort_values(["SampleID", "Depth_Read_Loci", "Total_Reads", "Run"],
                          ascending=[True, False, False, True])
stats["Rank"] = stats.groupby("SampleID").cumcount()
stats["Best_Run"] = (stats.Rank == 0).map({True: "yes", False: "no"})
ranks = stats.loc[stats.N_Runs > 1, ["SampleID", "Run", "Rank"]]
if not SHOW_ALL:
    stats = stats[stats.N_Runs > 1]

## 5. Screening table

| Column | Meaning |
|---|---|
| `SampleID` | Standardised ID |
| `Run` | Sequencing run (the sequencing ID if the allele table has no `Run` column) |
| `Raw_SampleID` | ID as written in the allele table |
| `N_Runs` | Number of runs the sample is in |
| `Total_Reads` | Reads over all loci |
| `Total_Loci` | Loci with at least one read |
| `Avg_Reads_per_Locus` | `Total_Reads` / `Total_Loci` |
| `Depth_Read_Loci` | Loci with more than `MIN_DEPTH` reads |
| `Best_Run` | `yes` / `no` (with `MERGE`: name of the best run) |
| `New_SampleID` | Only with `KEEP_DUPS`: the ID with its letter |

With `BEST_COVERAGE`, only the best row of each sample is kept. With `MERGE`, each sample has one row, and its runs are listed best first, separated by `;`.

In [ ]:
cols = ["SampleID", "Run", "Raw_SampleID", "N_Runs", "Total_Reads", "Total_Loci",
        "Avg_Reads_per_Locus", "Depth_Read_Loci", "Best_Run"]
if mode == "merge":
    # Add up each sample's reads per locus over its runs, then recount
    pooled = per_locus[per_locus.SampleID.isin(stats.SampleID)]
    pooled = pooled.groupby(["SampleID", "locus"], as_index=False).reads.sum()
    runs = stats.groupby("SampleID", sort=False).agg(
        Run=("Run", ";".join), Raw_SampleID=("Raw_SampleID", ";".join),
        N_Runs=("Run", "size"), Best_Run=("Run", "first")).reset_index()
    stats = summarise(pooled, ["SampleID"], MIN_DEPTH).merge(runs, on="SampleID").sort_values("SampleID")
elif mode == "best_coverage":
    stats = stats[stats.Rank == 0]
elif mode == "keep_dups":
    stats = stats.assign(New_SampleID=stats.SampleID + ("_" + stats.Rank.map(letter)).where(stats.N_Runs > 1, ""))
    cols.append("New_SampleID")
stats["Total_Reads"] = stats.Total_Reads.round().astype("int64")
# Files are read as latin-1 and written as latin-1, so every byte is kept as it was
stats[cols].to_csv(OUT_FILE, sep=separator(OUT_FILE), index=False, encoding="latin-1")

if run:
    print(f"Allele table: {len(ids)} IDs in {alleles.Run.nunique()} runs")
else:
    print(f"Allele table: {len(ids)} IDs; no Run column, so each sequencing ID counts as its own run")
print(f"Screened: {len(ids & samples)} samples (SampleType = sample) | not screened: "
      f"{len(ids & (set(man.SampleID) - samples))} controls or other types, {len(not_in_manifest)} not in the manifest")
if not_in_manifest:
    print(f"Not in the manifest, so not screened ({len(not_in_manifest)}): {shortlist(not_in_manifest)}")
if mixed:
    print(f"Note: {len(mixed)} IDs are 'sample' in one manifest row and another type in another; "
          f"screened as samples: {shortlist(mixed)}")
missing = sorted(samples - ids)
if missing:
    print(f"Manifest samples not in the allele table ({len(missing)}): {shortlist(missing)}")
shape = {"merge": "merged into one row each",
         "best_coverage": f"best run kept of {len(ranks)} sample-run rows",
         "keep_dups": f"{len(ranks)} sample-run rows, IDs ending in _a, _b, ..."}.get(mode, f"{len(ranks)} sample-run rows")
print(f"Samples in more than one run: {ranks.SampleID.nunique()} ({shape})")
print(f"Depth_Read_Loci = loci with more than {MIN_DEPTH:g} reads")
print(f"Written: {OUT_FILE}")

In [ ]:
stats[cols].reset_index(drop=True)

## 6. Allele table

This step runs only when `OUT_ALLELES_FILE` is set. The table has the same columns as the input. Samples in only one run, controls, and IDs not in the manifest are unchanged.

| Option | Change in the allele table |
|---|---|
| `BEST_COVERAGE` | Rows from the other runs are removed |
| `KEEP_DUPS` | All rows are kept, and IDs get `_a`, `_b`, ... |
| `MERGE` | One row per sample, locus and `asv`. Reads are summed and `norm.reads.locus` and `n.alleles` recalculated. `Run` lists all runs. |

In [ ]:
if not OUT_ALLELES_FILE:
    if mode:
        print("Note: only the screening table was written; set OUT_ALLELES_FILE to also write the allele table.")
else:
    out = resolve_alleles(al, {"id": sid, "locus": locus, "reads": reads, "asv": asv, "run": run,
                               "nida": find_col(al, None, ["nida"], ""),
                               "allele": find_col(al, None, ["allele"], ""),
                               "norm": find_col(al, None, ["norm.reads.locus"], ""),
                               "n_alleles": find_col(al, None, ["n.alleles"], "")},
                          links, repeated_rows, ranks, mode)
    out.to_csv(OUT_ALLELES_FILE, sep=separator(OUT_ALLELES_FILE), index=False, encoding="latin-1")
    print(f"Allele table rows: {len(al)} in, {len(out)} out")
    runs_out = (out[run] if run else out[sid]).str.strip()
    left = pd.DataFrame({"s": out[sid].map(normalise_id), "r": runs_out}).groupby("s").r.nunique()
    left = sorted(left[left > 1].index)
    if left:
        print(f"Note: {len(left)} IDs are still in more than one run in the allele table, because they were not "
              f"screened (controls or not in the manifest): {shortlist(left)}")
    print(f"Written: {OUT_ALLELES_FILE}")